# IMPORTANT
**Run this block of code for the notebook**
> If any error occurs with the block of code run the Requirements Section

In [1]:
import os
import subprocess
import serial
import time
import serial.tools.list_ports
import ipywidgets as widgets
from IPython.display import display, clear_output

serial_conn = None

def detect_serial_ports():
    available_ports = []

    ports = serial.tools.list_ports.comports()
    for port in ports:
        if any(x in port.device for x in ["ttyUSB", "ttyACM", "cu.usbmodem", "usbserial", "COM"]):
            available_ports.append(port.device)

    return available_ports

def connect_serial(port):
    global serial_conn
    try:
        serial_conn = serial.Serial(port, 921600, timeout=1)
        return True
    except Exception as e:
        return False

def send_command_string(command:str):
    if serial_conn and serial_conn.is_open:
        cmd = command+'\r\n'
        serial_conn.write(cmd.encode())

def send_command_string_with_response(command:str):
    global serial_conn

    if serial_conn and serial_conn.is_open:
        cmd = command+'\r\n'
        serial_conn.write(cmd.encode())
        time.sleep(0.3)
        try: 
            resp = serial_conn.read_all().decode(errors='ignore')
            return resp
        except:
            return 'Nan'
            
    else:
        return 'Nan'

def disconnect_serial():
    global serial_conn
    
    if serial_conn and serial_conn.is_open:
        serial_conn.close()
        serial_conn = None
        return True
    else: 
        return False


# Installing requirements
The following block of code installs the require modules for the notebook

In [ ]:
req_file = "requirements.txt"
if not os.path.exists(req_file):
    print("Please, contact with the trainers.")
else:
    print("Installing the requirements...")
    run_process_cmd(["pip", "install", "-r", req_file])

# Detect Minino
The next section works to detect if your selected port is a Minino device.

In [2]:
scan_button = widgets.Button(description="Scan Ports")
detect_button = widgets.Button(description="Detect Minino", icon="arrow-right")
dropdown_ports = widgets.Dropdown(description='Ports:', layout=widgets.Layout(width='20%'))
detect_minino_output = widgets.Output(layout={'border': '1px solid black'})

def scan_ports(b):
    dropdown_ports.options = detect_serial_ports()

def detect_minino(btn):
    
    if not connect_serial(dropdown_ports.value):
        with detect_minino_output:
            clear_output() 
            print("Error conecting the port")
        return
        
    data = send_command_string_with_response('h')
    
    if "minino" in data:
        with detect_minino_output:
            clear_output() 
            print(f"Minino Detected at {dropdown_ports.value}")
    else:
        with detect_minino_output:
            clear_output() 
            print("Minino no detected")
        
    disconnect_serial()

scan_button.on_click(scan_ports)
detect_button.on_click(detect_minino)

display(
    widgets.Box([scan_button, dropdown_ports, detect_button]),
    detect_minino_output
)

Box(children=(Button(description='Scan Ports', style=ButtonStyle()), Dropdown(description='Ports:', layout=Lay…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

# Hands on Lab #5 Wifi Deauthentication
## Learning Objectives
By the end of this lab, participants will be able to:
1. **Explain** the function of **deauthentication frames** in Wi‑Fi communications.
2. **Demonstrate** how deauthentication can be used to forcibly disconnect wireless clients.
3. **Capture** and **analyze** reassociation frames, probe requests, or handshake attempts following a deauthentication event.
4. **Identify** vulnerabilities in unprotected Wi‑Fi management frames that make these attacks possible.
5. **Perform** a targeted deauthentication attack using Minino against a specific device.
6. **Document** observed attack effects and **recommend** detection or mitigation strategies.

## What is Wifi Deauthenticator?
A Wi-Fi deauthentication attack is a type of denial-of-service attack that forces a device to disconnect from a wireless network by sending fake deauthentication frames

## Launching at Deauth Menu
To start the Wifi Deauthentication, first the Minino need to be configure. On the next code of section, it launch on Minino the Deauth Menu where you can set the respective configuration

In [8]:
connect_button = widgets.Button(description='Connect')
deauth_menu = widgets.Button(description='Go to Deauth Menu')
display_error = widgets.Output(layout={'border': '1px solid black'})

is_connected = False

def button_function(btn):
    global is_connected
    if not is_connected:
        if not connect_serial(dropdown_ports.value):
            with display_error:
                clear_output() 
                print("Error connecting the port")
            return
            
        time.sleep(2)
        connect_button.description = 'Disconnect'
        is_connected = True
        
    else:
        if not disconnect_serial():
            with display_error:
                clear_output() 
                print("Error disconnecting the port")
            return
            
        time.sleep(2)
        connect_button.description = 'Connect'
        is_connected = False

def go_to_deauth_menu(btn):
    global is_connected
    if not is_connected:
        return
    send_command_string("launch deauth")

connect_button.on_click(button_function)
deauth_menu.on_click(go_to_deauth_menu)

display(
    widgets.Box([connect_button, dropdown_ports]),
    deauth_menu,
    display_error
)

Box(children=(Button(description='Connect', style=ButtonStyle()), Dropdown(description='Ports:', index=1, layo…

Button(description='Go to Deauth Menu', style=ButtonStyle())

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

## Setting Up Minino
Once you have launched the Minino application, it will automatically scan for surrounding Wi-Fi Access Points (APs).

A menu will then appear, which you can navigate using the designated buttons.

The menu options are:

1. **Scan:** scan the Wi-Fi AP around (This step is often redundant as the networks are usually already listed, but the option is there to refresh the scan).

2. **Select:** Use this option to choose the specific Wi-Fi AP you intend to attack.

3. **Attack:** Here you will select the type of attack to perform. You have three available types: Broadcast, Rogue AP, and Combined.

4. **Run:** Selecting this final option will execute the chosen attack.